# Dataset Annotation Browser

`data/exemplars` 는 query dataset, 날짜 폴더는 gallery dataset 으로 보고 annotation JSON 과 bbox 를 함께 점검하는 노트북입니다.

- 큰 이미지는 비율을 유지한 채 축소해서 bbox 를 오버레이합니다.
- `ipywidgets` 기반 pagination 과 grid view 로 여러 샘플을 한 번에 훑어볼 수 있습니다.
- 아래 Query 셀과 Gallery 셀을 각각 실행하면 됩니다.


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

NOTEBOOK_ROOT = Path('/workspace/notebooks/dataset')

def resolve_data_root() -> Path:
    candidates = [NOTEBOOK_ROOT / 'data', Path.cwd() / 'data']
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


DATA_ROOT = resolve_data_root()
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}


In [2]:
def normalize_stem(path: Path) -> str:
    stem = path.stem
    return stem[:-5] if stem.endswith('_anno') else stem


def load_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def collect_dataset_records(dataset_root: Path, dataset_label: str) -> list[dict[str, Any]]:
    dataset_root = Path(dataset_root)
    images = {
        normalize_stem(path): path
        for path in dataset_root.rglob('*')
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    }
    annos = {
        normalize_stem(path): path
        for path in dataset_root.rglob('*.json')
        if path.is_file()
    }

    keys = sorted(set(images) | set(annos))
    records: list[dict[str, Any]] = []

    for key in keys:
        image_path = images.get(key)
        anno_path = annos.get(key)
        annotation = load_json(anno_path) if anno_path else None

        issues: list[str] = []
        if image_path is None:
            issues.append('missing_image')
        if anno_path is None:
            issues.append('missing_annotation')

        if image_path and anno_path and annotation:
            if annotation.get('img_name') and annotation['img_name'] != image_path.name:
                issues.append('img_name_mismatch')

            instances = annotation.get('instances') or []
            for idx, instance in enumerate(instances):
                bbox = instance.get('bbox') or {}
                coords = [bbox.get(axis) for axis in ('x1', 'y1', 'x2', 'y2')]
                if any(value is None for value in coords):
                    issues.append(f'instance_{idx}_bbox_missing')
                    continue
                x1, y1, x2, y2 = coords
                if not all(0 <= value <= 1 for value in coords):
                    issues.append(f'instance_{idx}_bbox_out_of_range')
                if x2 <= x1 or y2 <= y1:
                    issues.append(f'instance_{idx}_bbox_invalid_order')

        split_hint = image_path.parent.name if image_path else (anno_path.parent.name if anno_path else dataset_root.name)
        records.append(
            {
                'key': key,
                'dataset': dataset_label,
                'dataset_root': dataset_root,
                'split_hint': split_hint,
                'image_path': image_path,
                'anno_path': anno_path,
                'annotation': annotation,
                'issues': issues,
                'issue_text': ', '.join(issues) if issues else 'ok',
                'instance_count': len((annotation or {}).get('instances') or []),
            }
        )

    return records


def records_frame(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        rows.append(
            {
                'dataset': record['dataset'],
                'file': record['image_path'].name if record['image_path'] else record['key'],
                'folder': record['split_hint'],
                'instances': record['instance_count'],
                'issues': record['issue_text'],
                'image_path': str(record['image_path']) if record['image_path'] else '',
                'anno_path': str(record['anno_path']) if record['anno_path'] else '',
            }
        )
    columns = ['dataset', 'file', 'folder', 'instances', 'issues', 'image_path', 'anno_path']
    return pd.DataFrame(rows, columns=columns)


def summarize_records(records: list[dict[str, Any]], title: str) -> pd.DataFrame:
    frame = records_frame(records)
    print(f'{title}: {len(records)} pairs')
    if frame.empty:
        print('No records found.')
    else:
        print(frame['issues'].value_counts().rename_axis('issues').to_frame('count'))
    return frame


def discover_gallery_roots(data_root: Path) -> list[Path]:
    return sorted(
        path for path in data_root.iterdir()
        if path.is_dir() and path.name != 'exemplars'
    )


In [3]:
STATUS_COLORS = {
    'ACCEPTED': '#19a974',
    'UNREVIEWED': '#ff7f0e',
    'REJECTED': '#d62728',
}


def render_annotated_image(record: dict[str, Any], max_side: int = 480) -> Image.Image:
    if record['image_path'] is None:
        return Image.new('RGB', (max_side, max_side), color='white')

    with Image.open(record['image_path']) as image_obj:
        image = image_obj.convert('RGB')
    original_w, original_h = image.size
    scale = min(1.0, max_side / max(original_w, original_h))
    preview_size = (
        max(1, int(round(original_w * scale))),
        max(1, int(round(original_h * scale))),
    )
    preview = image.resize(preview_size, Image.Resampling.LANCZOS) if scale < 1.0 else image.copy()
    draw = ImageDraw.Draw(preview)

    annotation = record.get('annotation') or {}
    for idx, instance in enumerate(annotation.get('instances') or [], start=1):
        bbox = instance.get('bbox') or {}
        coords = [bbox.get(axis) for axis in ('x1', 'y1', 'x2', 'y2')]
        if any(value is None for value in coords):
            continue

        x1, y1, x2, y2 = coords
        left = max(0, min(preview.width - 1, x1 * preview.width))
        top = max(0, min(preview.height - 1, y1 * preview.height))
        right = max(left + 1, min(preview.width, x2 * preview.width))
        bottom = max(top + 1, min(preview.height, y2 * preview.height))

        status = instance.get('assignment_status') or 'UNKNOWN'
        color = STATUS_COLORS.get(status, '#1f77b4')
        label_name = instance.get('name') or instance.get('pet_id') or 'unknown'
        label = f"{idx}. {label_name} [{status}]"
        line_width = max(2, int(round(preview.width / 220)))
        draw.rectangle((left, top, right, bottom), outline=color, width=line_width)

        text_bbox = draw.textbbox((left, top), label)
        text_w = text_bbox[2] - text_bbox[0]
        text_h = text_bbox[3] - text_bbox[1]
        text_top = max(0, top - text_h - 6)
        text_right = min(preview.width, left + text_w + 8)
        draw.rectangle((left, text_top, text_right, text_top + text_h + 6), fill=color)
        draw.text((left + 4, text_top + 3), label, fill='white')

    return preview


def browse_records(records: list[dict[str, Any]], title: str, default_page_size: int = 8, default_columns: int = 4) -> widgets.VBox:
    records = list(records)
    if not records:
        return widgets.VBox([widgets.HTML(f'<b>{title}</b><br>No records found.')])

    title_html = widgets.HTML(f'<h3 style="margin:0 0 8px 0;">{title}</h3>')
    page_size = widgets.Dropdown(options=[4, 8, 12, 16], value=default_page_size, description='Page')
    columns = widgets.Dropdown(options=[2, 3, 4], value=default_columns, description='Cols')
    issue_filter = widgets.Dropdown(
        options=[('all', 'all'), ('only issues', 'only issues'), ('ok only', 'ok only')],
        value='all',
        description='Filter'
    )
    search = widgets.Text(value='', description='Search')
    prev_btn = widgets.Button(description='Prev', icon='arrow-left')
    next_btn = widgets.Button(description='Next', icon='arrow-right')
    page_label = widgets.HTML()
    out = widgets.Output()

    state = {'page': 1}

    def filtered_records() -> list[dict[str, Any]]:
        subset = records
        if issue_filter.value == 'only issues':
            subset = [record for record in subset if record['issues']]
        elif issue_filter.value == 'ok only':
            subset = [record for record in subset if not record['issues']]

        keyword = search.value.strip().lower()
        if keyword:
            subset = [
                record for record in subset
                if keyword in (record['image_path'].name.lower() if record['image_path'] else '')
                or keyword in record['split_hint'].lower()
                or keyword in json.dumps(record.get('annotation') or {}, ensure_ascii=False).lower()
            ]
        return subset

    def redraw(*_args: Any) -> None:
        subset = filtered_records()
        total = len(subset)
        size = page_size.value
        total_pages = max(1, math.ceil(total / size))
        state['page'] = min(max(1, state['page']), total_pages)
        start = (state['page'] - 1) * size
        chunk = subset[start:start + size]

        page_label.value = f'<b>{state["page"]}</b> / {total_pages} &nbsp; ({total} items)'
        prev_btn.disabled = state['page'] <= 1
        next_btn.disabled = state['page'] >= total_pages

        with out:
            out.clear_output(wait=True)
            if not chunk:
                print('No records match the current filter.')
                return

            cols = columns.value
            rows = math.ceil(len(chunk) / cols)
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.6, rows * 4.6))
            axes = list(axes.flat) if hasattr(axes, 'flat') else [axes]

            for ax, record in zip(axes, chunk):
                ax.imshow(render_annotated_image(record, max_side=520))
                title_lines = [record['image_path'].name if record['image_path'] else record['key']]
                title_lines.append(f"instances={record['instance_count']}")
                if record['issues']:
                    title_lines.append(record['issue_text'])
                ax.set_title('\n'.join(title_lines), fontsize=10)
                ax.axis('off')

            for ax in axes[len(chunk):]:
                ax.axis('off')

            plt.tight_layout()
            plt.show()

    def go_prev(_button: widgets.Button) -> None:
        state['page'] -= 1
        redraw()

    def go_next(_button: widgets.Button) -> None:
        state['page'] += 1
        redraw()

    for control in (page_size, columns, issue_filter):
        control.observe(lambda change: redraw() if change['name'] == 'value' else None, names='value')
    search.observe(lambda change: redraw() if change['name'] == 'value' else None, names='value')
    prev_btn.on_click(go_prev)
    next_btn.on_click(go_next)

    redraw()
    controls = widgets.HBox([page_size, columns, issue_filter, search, prev_btn, next_btn, page_label])
    return widgets.VBox([title_html, controls, out])


## Query View

`data/exemplars` 전체를 query dataset 으로 로드합니다. 폴더명은 개체명 힌트로 그대로 유지됩니다.


In [ ]:
QUERY_ROOT = DATA_ROOT / 'exemplars'
query_records = collect_dataset_records(QUERY_ROOT, dataset_label='query')
query_frame = summarize_records(query_records, title='Query dataset')
display(query_frame.head(20))
display(browse_records(query_records, title='Query Annotation Browser', default_page_size=8, default_columns=4))


Query dataset: 468 pairs
                   count
issues                  
img_name_mismatch    468


,dataset,file,folder,instances,issues,image_path,anno_path
0,query,exemplars_가을_KakaoTalk_20260305_191405350_01.jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
1,query,exemplars_가을_img.jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
2,query,exemplars_가을_img (5).jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
3,query,exemplars_가을_img_640_resize (45).jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
4,query,exemplars_가을_가을.jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
5,query,exemplars_가을_가을 (2).jpg,가을,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/가을...,/workspace/notebooks/dataset/data/exemplars/가을...
6,query,exemplars_금동_금동 (1).jpg,금동,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/금동...,/workspace/notebooks/dataset/data/exemplars/금동...
7,query,exemplars_금동_금동 (2).jpg,금동,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/금동...,/workspace/notebooks/dataset/data/exemplars/금동...
8,query,exemplars_금동_금동 (3).jpg,금동,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/금동...,/workspace/notebooks/dataset/data/exemplars/금동...
9,query,exemplars_금동_금동 (4).jpg,금동,1,img_name_mismatch,/workspace/notebooks/dataset/data/exemplars/금동...,/workspace/notebooks/dataset/data/exemplars/금동...


## Gallery View

기본값으로 `data` 아래 첫 번째 날짜 폴더를 선택합니다. 다른 날짜를 보고 싶으면 `GALLERY_ROOT` 값을 바꿔 다시 실행하면 됩니다.


In [5]:
gallery_roots = discover_gallery_roots(DATA_ROOT)
gallery_roots


[PosixPath('/workspace/notebooks/dataset/data/2026-03-30')]

In [6]:
GALLERY_ROOT = gallery_roots[0]
gallery_records = collect_dataset_records(GALLERY_ROOT, dataset_label=f'gallery:{GALLERY_ROOT.name}')
gallery_frame = summarize_records(gallery_records, title=f'Gallery dataset ({GALLERY_ROOT.name})')
display(gallery_frame.head(20))
display(browse_records(gallery_records, title=f'Gallery Annotation Browser: {GALLERY_ROOT.name}', default_page_size=12, default_columns=4))


Gallery dataset (2026-03-30): 238 pairs
        count
issues       
ok        238


,dataset,file,folder,instances,issues,image_path,anno_path
0,gallery:2026-03-30,1 (1).jpg,2026-03-30,1,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
1,gallery:2026-03-30,1 (10).jpg,2026-03-30,1,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
2,gallery:2026-03-30,1 (11).jpg,2026-03-30,2,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
3,gallery:2026-03-30,1 (12).jpg,2026-03-30,1,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
4,gallery:2026-03-30,1 (13).jpg,2026-03-30,2,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
5,gallery:2026-03-30,1 (14).jpg,2026-03-30,1,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
6,gallery:2026-03-30,1 (15).jpg,2026-03-30,2,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
7,gallery:2026-03-30,1 (16).jpg,2026-03-30,2,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
8,gallery:2026-03-30,1 (17).jpg,2026-03-30,2,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
9,gallery:2026-03-30,1 (18).jpg,2026-03-30,4,ok,/workspace/notebooks/dataset/data/2026-03-30/1...,/workspace/notebooks/dataset/data/2026-03-30/1...
